---
title: '```{python}'
---

In [ ]:
from detectgptmodel.fastgpt import FastGptDetect, get_sampling_discrepancy, calculate_ttr, calc_punctuation,calculate_sentence_length_std
import polars as pl
import polars.selectors as cs
import numpy as np
from textblob import TextBlob
from tqdm import tqdm
import textstat
import seaborn as sns
import joblib
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from scipy.stats import loguniform
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn import tree
from sklearn.decomposition import PCA

In [ ]:
detect_gpt = FastGptDetect(model="roberta-large-mnli")

In [ ]:
N_SAMPLE_SIZE = 1
dolly_df = pl.read_parquet("data/dolly_response.parquet")
xsum_df = pl.read_parquet("data/xsum.parquet").rename({'document': 'text'})['text', ].with_columns(pl.lit(False).alias("Is AI")) \
.filter(~pl.col("text").str.split(' ').list.len()> 200)[:N_SAMPLE_SIZE]
df_gpt = dolly_df.sample(shuffle=True, n=N_SAMPLE_SIZE)

df = df_gpt.extend(xsum_df)
df = df.sample(shuffle=True, fraction=1)
df

In [ ]:
def sample_text(text: str):
  return detect_gpt.predict(text)

In [ ]:
results = [sample_text(text) for text in tqdm(df['text'].to_list(), desc="Processing", position=0, leave=True)]

# 2. Attach the results back to the DataFrame as a new Series
df_fast = df.with_columns(
    fast_gpt = pl.Series(results, dtype=pl.List(pl.Float32))
)

df_fast

In [ ]:
df_fast = pl.read_parquet("data/roberta_nmli_train.parquet")
df_fast

In [ ]:
def get_base_features(text: str):
  return detect_gpt.get_base_features(text)

results = np.array([get_base_features(text) for text in tqdm(df_fast['text'].to_list(), desc="Processing", position=0, leave=True)])

df_fast = df_fast.with_columns(
  ttr = results[:, 0],
  punct = results[:, 1],
  sentence_std = results[:, 2],
  polarity = results[:, 3],
  subjectivity = results[:, 4],
  flesch = results[:, 5],
  gunning_fog = results[:, 6],
)

df_fast

# df_fast = pl.read_parquet("data/roberta_nmli_train.parquet")
# df_fast
# ```

In [ ]:
max_len_1 = df_fast.select(pl.col("fast_gpt").list.len().max()).item()
df_fast_flat = df_fast.with_columns(
  [pl.col("fast_gpt").list.get(i).alias(f"fast_gpt_1_{i+1}") for i in range(max_len_1)],
).drop(["fast_gpt"]).filter(
    pl.all_horizontal(cs.numeric().is_finite())
)

df_fast_flat

In [ ]:
# X = df.drop(["text", "Is AI", "punct", 'polarity', 'subjectivity', 'flesch', 'gunning_fog']).to_numpy().astype(np.float32)
X = df_fast_flat.drop(["text", "Is AI"]).to_numpy()
y = df_fast_flat["Is AI"].cast(pl.Float32).to_numpy()

In [ ]:
def train_model(model, rocs, accs, f1s, recalls):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)

  model.fit(X_train, y_train)

  # Predict and Evaluate
  proba_predictions = model.predict_proba(X_test)[:, 1]
  predictions = model.predict(X_test)

  # print(f"ROC AUC: {roc_auc_score(y_test, proba_predictions):.2%}")
  # print(f"Accuracy: {accuracy_score(y_test, predictions):.2%}")
  rocs.append(roc_auc_score(y_test, proba_predictions))
  accs.append(accuracy_score(y_test, predictions))
  f1s.append(f1_score(y_test, predictions))
  recalls.append(recall_score(y_test, predictions))

rocs = []
accs = []
f1s = []
recalls = []
for i in tqdm(range(100)):
  model = Pipeline([
    ('scaler', RobustScaler()),
    # ('scaler', StandardScaler()),
    # ('pca', PCA(n_components=0.95)), # Keep 95% of variance
    ('clf', SVC(kernel='rbf', C=4, probability=True, gamma='scale'))
  ])

  # model = Pipeline([
  #   ('scaler', StandardScaler()),
  #   # ('pca', PCA(n_components=0.95)), # Keep 95% of variance
  #   ('clf', tree.DecisionTreeClassifier())
  # ])

  # model = Pipeline([
  #   # ('scaler', MinMaxScaler()),
  #   ('scaler', StandardScaler()),
  #   # ('pca', PCA(n_components=0.95)), # Keep 95% of variance
  #   ('clf', LogisticRegression(max_iter=300))
  # ])

  # model = Pipeline([
  #   ('scaler', StandardScaler()),
  #   # ('clf', GradientBoostingClassifier(
  #   #     n_estimators=500,        # High number of trees, balanced by early stopping below
  #   #     learning_rate=0.01,      # Lower learning rate requires more estimators, yields robust results
  #   #     max_depth=6,             # Allows for complex multi-feature interactions
  #   #     subsample=0.8,           # Stochastic Gradient Boosting: Uses 80% of data per tree
  #   #     max_features='sqrt',     # Considers approx 4-5 features per split (sqrt of 20)
  #   #     validation_fraction=0.1, # Sets aside 10% of training data for early stopping
  #   #     n_iter_no_change=20,     # Stops training if validation score doesn't improve for 20 trees
  #   #     random_state=420         # Seed for reproducibility
  #   # ))
  #   ('clf', RandomForestClassifier(
  #       n_estimators=500,         # Total number of trees in the forest
  #       max_depth=6,              # Limits depth to prevent overfitting
  #       max_features='sqrt',      # Randomly selects sqrt(total_features) at each split
  #       bootstrap=True,           # Uses bootstrap samples (similar concept to subsample)
  #       n_jobs=-1,                # Optional: Uses all CPU cores to speed up training
  #       random_state=420          # Seed for reproducibility
  #   ))
  # ])

  train_model(model, rocs, accs, f1s, recalls)

In [ ]:
joblib.dump(model, 'models/_model_gradient__.pkl')

In [ ]:
print(np.mean(rocs))
sns.histplot(rocs)
# plt.savefig('rocs_models_gpt_2.pdf')
# plt.savefig('rocs_models_dolly.pdf')

In [ ]:
def calculate_percentage_averages(data):
    # Convert input to a numpy array of floats
    arr = np.array(data, dtype=float)
    
    # Handle empty arrays
    if arr.size == 0:
        return "List is empty."

    # 1. Arithmetic Mean
    arithmetic_mean = np.mean(arr)

    # 2. Median
    median_val = np.median(arr)

    # 3. Logit Transformation Mean
    # Convert to decimal. np.clip() safely caps all values at once to prevent math errors.
    p = np.clip(arr / 100.0, 1e-5, 0.99999)
    
    # Vectorized Logit: Calculates ln(p / (1-p)) for the entire array instantly
    logits = np.log(p / (1.0 - p))
    avg_logit = np.mean(logits)
    
    # Inverse logit (expit) back to percentage
    logit_mean = (np.exp(avg_logit) / (1.0 + np.exp(avg_logit))) * 100

    # 4. Complement Geometric Mean
    # Calculate defect rates and clip to a minimum of 0.001 to prevent log(0)
    complements = np.clip(100.0 - arr, 0.001, None)
    
    # Vectorized Geometric Mean calculation
    geom_mean_complement = np.exp(np.mean(np.log(complements)))
    comp_geom_mean = 100.0 - geom_mean_complement

    # Return rounded results (converted to native python floats for clean dictionary output)
    return {
        "Arithmetic Mean": round(float(arithmetic_mean), 4),
        "Median": round(float(median_val), 4),
        "Logit Mean": round(float(logit_mean), 4),
        "Complement Geometric Mean": round(float(comp_geom_mean), 4)
    }

In [ ]:
calculate_percentage_averages(rocs)
# calculate_percentage_averages(f1s)

In [ ]:
df_score = pl.DataFrame({
  'roc': rocs,
  'acc': accs,
  'f1': f1s,
  'recall': recalls
})
# df_score.write_csv('dolly_model_score.csv')
df_score

In [ ]:
import matplotlib.pyplot as plt

mean_roc = np.mean(rocs)
std_roc = np.std(rocs)

sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 5))

# 3. Create the histogram with a Kernel Density Estimate (KDE) curve
sns.histplot(rocs, kde=True, bins=20, color="steelblue", edgecolor="black")

# 4. Add a vertical line to show the mean visually
plt.axvline(mean_roc, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_roc:.4f}')

# 5. Add titles, labels, and legend
plt.title("Distribution of ROC AUC Scores", fontsize=14, pad=15)
plt.xlabel("ROC AUC Score", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend()
plt.savefig("roc-score Detect AI.pdf")